In [122]:
import pandas as pd

In [123]:
df = pd.read_excel('Sprint_History.xlsx', sheet_name='Raw Data')

In [124]:
df

,Work Item Type,ID,Title,State,Iteration Path,Resolved Date,Story Points,Activated Date,Created Date,PICategory
0,User Story,485934,Configure the Casting Model in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:47:08,2.0,2025-04-08 14:11:07,2025-03-20 09:48:37,Modelling & Configuration
1,User Story,485932,Configure the Final Packaging Model in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:47:35,1.0,2025-04-08 14:11:00,2025-03-20 09:47:16,Modelling & Configuration
2,User Story,485927,Master Data Configuration for Pouching in MES,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 10:48:37,1.0,2025-04-08 14:10:53,2025-03-20 09:37:17,Modelling & Configuration
3,User Story,485930,Configure Site Model for HA-CMC Powder & Raw M...,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 12:59:56,1.0,2025-04-09 10:48:12,2025-03-20 09:43:10,Modelling & Configuration
4,User Story,485924,Execute the Weighing Process - IoT W&D,Resolved,DSM-Firmenich\Sprint 01,2025-04-09 16:11:27,8.0,2025-03-27 14:04:13,2025-03-20 09:27:29,Integration - Equipment / Connect IoT
...,...,...,...,...,...,...,...,...,...,...
370,User Story,579449,Sprint 22 - Release Management,Resolved,DSM-Firmenich\Sprint 22,2026-06-02 12:00:22,NaN,2026-06-02 09:17:31,2026-05-21 09:32:17,Release Management
371,User Story,581720,Sprint 23 - Release Management,Resolved,DSM-Firmenich\Sprint 23,2026-06-15 17:27:21,NaN,2026-06-15 17:27:21,2026-06-05 11:02:02,Release Management
372,User Story,562436,"[ENHANCEMENT] Improve EBR - Part I - Generic, ...",Resolved,DSM-Firmenich\Sprint 23,2026-06-17 16:56:19,13.0,2026-06-09 08:33:37,2026-02-10 13:07:44,Customization - Business Logic
373,Bug,581548,Combine with Partial Track-Outs,Resolved,DSM-Firmenich\Sprint 24,2026-06-24 13:40:50,5.0,2026-06-24 09:13:49,2026-06-03 16:48:58,Customization - Business Logic


In [125]:
df['Sprint'] = df['Iteration Path'].apply(lambda x: x.split('\\')[-1])
df['Story Points'] = df['Story Points'].fillna(0).astype(int)
df = df[df['State'].isin(['Resolved', 'Done', 'Closed'])]

In [126]:
# Sprint Summary
import datetime as dt
sprint_df = df.groupby('Sprint')[['Story Points']].sum().reset_index()
sprint_df['Items Resolved'] = df.groupby('Sprint')['ID'].count().values
sprint_df['Last Resolved Date'] = df.groupby('Sprint')['Resolved Date'].max().values
sprint_df['Recency Rank'] = sprint_df['Last Resolved Date'].rank(ascending=False, method='min').astype(int)
# sprint_df['Last Resolved Date'] = sprint_df['Last Resolved Date'].dt.date.astype('datetime64[ns]')
sprint_df

,Sprint,Story Points,Items Resolved,Last Resolved Date,Recency Rank
0,Sprint 01,18,7,2025-08-22 17:38:46,17
1,Sprint 02,53,19,2025-07-07 09:49:15,22
2,Sprint 03,39,30,2025-07-13 20:18:29,21
3,Sprint 04,38,23,2025-05-22 14:55:59,24
4,Sprint 05,23,9,2025-06-04 18:54:06,23
5,Sprint 06,37,19,2025-08-06 14:53:12,19
6,Sprint 07,34,27,2025-07-13 20:19:50,20
7,Sprint 08,47,20,2025-12-02 10:00:59,9
8,Sprint 09,38,28,2025-12-02 10:00:59,9
9,Sprint 10,35,16,2025-08-14 13:21:44,18


In [127]:
# Exclude latest sprints from sampling
exclude_from_sampling = 1
sprint_df = sprint_df[sprint_df['Recency Rank'] > exclude_from_sampling]
sprint_df

,Sprint,Story Points,Items Resolved,Last Resolved Date,Recency Rank
0,Sprint 01,18,7,2025-08-22 17:38:46,17
1,Sprint 02,53,19,2025-07-07 09:49:15,22
2,Sprint 03,39,30,2025-07-13 20:18:29,21
3,Sprint 04,38,23,2025-05-22 14:55:59,24
4,Sprint 05,23,9,2025-06-04 18:54:06,23
5,Sprint 06,37,19,2025-08-06 14:53:12,19
6,Sprint 07,34,27,2025-07-13 20:19:50,20
7,Sprint 08,47,20,2025-12-02 10:00:59,9
8,Sprint 09,38,28,2025-12-02 10:00:59,9
9,Sprint 10,35,16,2025-08-14 13:21:44,18


In [ ]:
# Construct a DF of story points for each sprint, for 1000 simulations
import numpy as np

target_type = 'Items Resolved'  # 'Items Resolved' or 'Story Points'
simulations = 1000
max_number_sprints = 40

columns: list[str] = [f'S{i+1}' for i in range(max_number_sprints)]

values = sprint_df[target_type].to_numpy()
# It's more performant to generate all random values at once, then reshape into a 2D array, than to generate each row separately.
draws = np.random.choice(values, size=(simulations, max_number_sprints))
draws.cumsum(axis=1)
simulation_df = pd.DataFrame(draws.cumsum(axis=1), columns=columns)
simulation_df

,S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,...,S31,S32,S33,S34,S35,S36,S37,S38,S39,S40
0,17,24,44,59,66,89,106,125,144,163,...,594,618,634,638,641,671,688,691,710,731
1,19,26,28,30,62,89,105,120,125,145,...,496,519,538,555,558,560,592,607,634,654
2,4,23,42,59,82,84,95,110,129,132,...,440,472,504,506,508,511,522,550,582,599
3,11,15,36,57,66,85,113,132,139,158,...,501,525,552,554,558,562,581,583,603,612
4,28,30,60,87,108,132,151,178,208,229,...,618,634,636,641,668,691,698,717,719,730
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,9,18,50,74,85,115,120,127,142,161,...,521,536,568,587,592,612,615,632,662,664
996,20,27,44,46,69,71,90,111,134,164,...,496,528,537,544,565,568,583,594,613,624
997,27,43,66,89,93,123,142,144,151,168,...,475,477,496,515,547,564,573,576,579,603
998,9,28,47,66,81,98,103,106,136,138,...,459,489,505,537,554,570,594,617,640,642


In [150]:
target = 5 if target_type == 'Items Resolved' else 30 # Simulation target

targets_df = simulation_df >= target
first_hit = targets_df.idxmax(axis=1)
first_hit[~targets_df.any(axis=1)] = None
first_hit.value_counts().sort_index()

S1    826
S2    168
S3      6
Name: count, dtype: int64